# Data Quality Monitoring with Great Expectations

In [ ]:
import great_expectations as gx
import pandas as pd

In [ ]:
df = pd.read_csv("../data/sales.csv",delimiter=",")
df

## Create an Context for GX

In [ ]:
context = gx.get_context()

In [ ]:
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})


In [ ]:
suite_name = "sales_data_quality_suite"
suite = gx.ExpectationSuite(name=suite_name)

ex_work = gx.expectations.ExpectColumnValuesToBeBetween(
    column="work experience", min_value=0, max_value=14, severity="warning"
)
suite.add_expectation(ex_work)
ex_age = gx.expectations.ExpectColumnValuesToBeBetween(
    column="training level", min_value=0, max_value=3, severity="critical"
)
suite.add_expectation(ex_age)
ex_salary = gx.expectations.ExpectColumnValuesToBeBetween(
    column="salary", min_value=30000, max_value=120000, severity="critical"
)
suite.add_expectation(ex_salary)
ex_sales = gx.expectations.ExpectColumnValuesToBeBetween(
    column="sales", min_value=20000, max_value=80000, severity="warning"
)
suite.add_expectation(ex_sales)
ex_sales_dist = gx.expectations.ExpectColumnKLDivergenceToBeLessThan(
    column="sales",
    partition_object={
        "bins": [0,20000, 30000, 40000, 50000, 60000, 70000, 80000],
        "weights": [0.1, 0.15,0.25 ,0.25,0.25, 0.0,0.0],
    },
    threshold=0.1,
)
suite.add_expectation(ex_sales_dist)
ex_salary_zscore = gx.expectations.ExpectColumnValueZScoresToBeLessThan(
    column="salary", threshold=3, double_sided=True
)
suite.add_expectation(ex_salary_zscore)


validation_result = batch.validate(suite)
print(validation_result)
